In [1]:
import os

from dotenv import load_dotenv
_ = load_dotenv()

TPCDS_TOOLS_DIR = os.getenv("TPCDS_TOOLS_DIR")
DATA_DIR = os.path.join(TPCDS_TOOLS_DIR, "data")

print(f"TPCDS_TOOLS_DIR: {TPCDS_TOOLS_DIR}")
print(f"DATA_DIR: {DATA_DIR}")

TPCDS_TOOLS_DIR: /home/basbenja/Documents/DEAI/data-warehouses/project/part1/tpcds-tools
DATA_DIR: /home/basbenja/Documents/DEAI/data-warehouses/project/part1/tpcds-tools/data


In [2]:
print(os.listdir(DATA_DIR))

['household_demographics.dat', 'warehouse.dat', 'web_page.dat', 'income_band.dat', 'web_site.dat', 'ship_mode.dat', 'web_sales.dat', 'date_dim.dat', 'inventory.dat', 'dbgen_version.dat', 'item.dat', 'catalog_sales.dat', 'customer.dat', 'store_sales.dat', 'web_returns.dat', 'promotion.dat', 'catalog_returns.dat', 'customer_demographics.dat', 'call_center.dat', 'store.dat', 'reason.dat', 'time_dim.dat', 'customer_address.dat', 'store_returns.dat', 'catalog_page.dat']


## **Starting Point: SparkSession**
The entry point into all functionality in Spark is the `SparkSession` class. To create a basic `SparkSession`, just use `SparkSession.builder`.

Source: [Spark SQL Guide - Getting started](https://spark.apache.org/docs/latest/sql-getting-started.html)

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("tpcds") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/22 16:56:35 WARN Utils: Your hostname, bbas, resolves to a loopback address: 127.0.1.1; using 10.66.79.220 instead (on interface wlp0s20f3)
26/09/22 16:56:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 16:56:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


- [Spark SQL - SQL Syntax](https://spark.apache.org/docs/latest/sql-ref-syntax.html)
- [Spark SQL - Data Types](https://spark.apache.org/docs/latest/sql-ref-datatypes.html)

In [4]:
path = "/home/basbenja/Documents/DEAI/data-warehouses/project/part1/tpcds-tools/tpcds.sql"
with open(path, "r") as f:
    sql = f.read()
    # This will throw an error becuase the SQL file contains multiple statements,
    # and spark.sql only allows one statement at a time.
    try:
        spark.sql(sql)
    except Exception as e:
        print(f"Error executing SQL: {e}")

Error executing SQL: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'create': extra input 'create'. SQLSTATE: 42601 (line 44, pos 0)



In [5]:
try:
    spark.sql("""create table dbgen_version
    (
        dv_version                varchar(16)                   ,
        dv_create_date            date                          ,
        dv_create_time            time                          ,
        dv_cmdline_args           varchar(200)
    );""")
except Exception as e:
    print(f"Error creating table dbgen_version: {e}")

Error creating table dbgen_version: [UNSUPPORTED_TIME_TYPE] The data type TIME is not supported. SQLSTATE: 0A000


In [6]:
df = spark.read.csv(
    path=os.path.join(DATA_DIR, "warehouse.dat"),
    sep='|'
)
print(type(df))
# We see that the data has been loaded correctly but it has no schema (all columns
# are unnamed)
df.show()

<class 'pyspark.sql.classic.dataframe.DataFrame'>
+---+----------------+--------------------+------+----+----------+-------+---------+--------+-----------------+----+-----+-------------+----+----+
|_c0|             _c1|                 _c2|   _c3| _c4|       _c5|    _c6|      _c7|     _c8|              _c9|_c10| _c11|         _c12|_c13|_c14|
+---+----------------+--------------------+------+----+----------+-------+---------+--------+-----------------+----+-----+-------------+----+----+
|  1|AAAAAAAABAAAAAAA| Conventional childr|977787| 651|      6th |Parkway|Suite 470|Fairview|Williamson County|  TN|35709|United States|  -5|NULL|
|  2|AAAAAAAACAAAAAAA|Important issues liv|138504| 600|View First| Avenue|  Suite P|Fairview|Williamson County|  TN|35709|United States|  -5|NULL|
|  3|AAAAAAAADAAAAAAA|         Doors canno|294242| 534|Ash Laurel|    Dr.|  Suite 0|Fairview|Williamson County|  TN|35709|United States|  -5|NULL|
|  4|AAAAAAAAEAAAAAAA|Bad cards must make.|621234| 368|Wilson Elm|  

Now, we will try load the schema of this data and associate it.

In [7]:
sql_schema_path = os.path.join(TPCDS_TOOLS_DIR, "tpcds.sql")
with open(sql_schema_path, "r") as f:
    sql_schema = f.read()

no_comments = "\n".join(
    line for line in sql_schema.splitlines() if not line.strip().startswith("--")
)
tables = [table.strip() for table in no_comments.split(";") if table.strip()]

for t in tables:
    if "create table warehouse" in t.lower():
        print(t)

create table warehouse
(
    w_warehouse_sk            integer               not null,
    w_warehouse_id            char(16)              not null,
    w_warehouse_name          varchar(20)                   ,
    w_warehouse_sq_ft         integer                       ,
    w_street_number           char(10)                      ,
    w_street_name             varchar(60)                   ,
    w_street_type             char(15)                      ,
    w_suite_number            char(10)                      ,
    w_city                    varchar(60)                   ,
    w_county                  varchar(30)                   ,
    w_state                   char(2)                       ,
    w_zip                     char(10)                      ,
    w_country                 varchar(20)                   ,
    w_gmt_offset              decimal(5,2)                  ,
    primary key (w_warehouse_sk)
)


In [8]:
spark.sql("""create table warehouse
(
    w_warehouse_sk            integer               not null,
    w_warehouse_id            char(16)              not null,
    w_warehouse_name          varchar(20)                   ,
    w_warehouse_sq_ft         integer                       ,
    w_street_number           char(10)                      ,
    w_street_name             varchar(60)                   ,
    w_street_type             char(15)                      ,
    w_suite_number            char(10)                      ,
    w_city                    varchar(60)                   ,
    w_county                  varchar(30)                   ,
    w_state                   char(2)                       ,
    w_zip                     char(10)                      ,
    w_country                 varchar(20)                   ,
    w_gmt_offset              decimal(5,2)
)""")

DataFrame[]

In [9]:
spark.sql(""" show tables; """).show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default|warehouse|      false|
+---------+---------+-----------+



## Schema-on-read
Instead of reading the `.dat` file as untyped/unnamed columns and casting afterwards,
we grab the schema Spark already parsed from our `create table warehouse (...)`
statement (it's sitting in the catalog now) and pass it directly to `spark.read.csv`.
This way Spark names and types each column while parsing, in one pass.

Note: the TPC-DS `.dat` files end every row with a trailing `|`, so there's one extra
empty field after `w_gmt_offset`. We need to confirm below whether the schema (14
fields) vs. the raw row shape (15 fields) causes a problem, or whether Spark just
truncates to the schema length.

In [10]:
warehouse_schema = spark.table("warehouse").schema
print(warehouse_schema)

StructType([StructField('w_warehouse_sk', IntegerType(), True), StructField('w_warehouse_id', StringType(), True), StructField('w_warehouse_name', StringType(), True), StructField('w_warehouse_sq_ft', IntegerType(), True), StructField('w_street_number', StringType(), True), StructField('w_street_name', StringType(), True), StructField('w_street_type', StringType(), True), StructField('w_suite_number', StringType(), True), StructField('w_city', StringType(), True), StructField('w_county', StringType(), True), StructField('w_state', StringType(), True), StructField('w_zip', StringType(), True), StructField('w_country', StringType(), True), StructField('w_gmt_offset', DecimalType(5,2), True)])


In [11]:
warehouse_df = spark.read.csv(
    path=os.path.join(DATA_DIR, "warehouse.dat"),
    sep='|',
    schema=warehouse_schema,
)
print(f"Row count: {warehouse_df.count()}")
warehouse_df.printSchema()
warehouse_df.show()

Row count: 5
root
 |-- w_warehouse_sk: integer (nullable = true)
 |-- w_warehouse_id: string (nullable = true)
 |-- w_warehouse_name: string (nullable = true)
 |-- w_warehouse_sq_ft: integer (nullable = true)
 |-- w_street_number: string (nullable = true)
 |-- w_street_name: string (nullable = true)
 |-- w_street_type: string (nullable = true)
 |-- w_suite_number: string (nullable = true)
 |-- w_city: string (nullable = true)
 |-- w_county: string (nullable = true)
 |-- w_state: string (nullable = true)
 |-- w_zip: string (nullable = true)
 |-- w_country: string (nullable = true)
 |-- w_gmt_offset: decimal(5,2) (nullable = true)

+--------------+----------------+--------------------+-----------------+---------------+-------------+---------------+--------------+--------+-----------------+-------+----------+-------------+------------+
|w_warehouse_sk|  w_warehouse_id|    w_warehouse_name|w_warehouse_sq_ft|w_street_number|w_street_name|  w_street_type|w_suite_number|  w_city|         w_co